# UdaPlay Agent Demonstration

This notebook demonstrates the execution of the UdaPlay AI Research Agent, testing its RAG capabilities, web search fallback, and conversational memory.

In [1]:
import logging
import json
import os
import sys

# 1. Add the parent directory to the system path so Jupyter can find 'src'
sys.path.append(os.path.abspath('..'))

# Configure logging to see the agent's thought process in the notebook
logging.basicConfig(level=logging.INFO, stream=sys.stdout, format='%(asctime)s - %(levelname)s - %(message)s')

from src.agent import UdaPlayAgent

agent = UdaPlayAgent()
print("UdaPlay Agent successfully initialized!")

c:\Users\andres.leonrangel\ws\gh-aleon1220\udaplay-ai-research-agent\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


2026-03-30 16:51:11,343 - INFO - Initialized ChromaDB collection: 'games' at 'vector_db'
UdaPlay Agent successfully initialized!


### Helper Functions
To format and cleanly print the output responses from the agent.

In [ ]:
def print_agent_response(result: dict):
    """
    Parses and prints the agent's response cleanly to the notebook output.
    """
    print("=== UDA-PLAY AGENT RESPONSE ===\n")
    
    # 1. Print the Natural Language Answer
    print("[Natural Language Answer]")
    print(result.get("final_answer", "No answer generated."))
    print("\n-------------------------------\n")
    
    # 2. Print the Structured Facts
    print("[Extracted Structured Data]")
    structured_data = result.get("structured_response", [])
    
    if not structured_data:
        print("No facts were extracted.")
    else:
        for i, fact in enumerate(structured_data, 1):
            # Check if it's a Pydantic model instance
            if hasattr(fact, 'attribute') and hasattr(fact, 'value'):
                print(f"  {i}. {fact.attribute}: {fact.value}")
            # Check if it was parsed as a standard dictionary
            elif isinstance(fact, dict):
                print(f"  {i}. {fact.get('attribute', 'Unknown')}: {fact.get('value', 'Unknown')}")
            # Fallback for unexpected formats
            else:
                print(f"  {i}. {fact}")
                
    print("\n===============================")

# simplified version of the print function focuses on the core output without the extra formatting and error handling. 
# It assumes that the agent's response is always in the expected format, which may not be the case in practice, serves as a clear example of how to display the results.
def print_agent_response_simple(result):
    print("\n" + "="*50)
    print("FINAL ANSWER (Natural Language):")
    print("-"*50)
    print(result['final_answer'])
    print("\n" + "="*50)
    print("STRUCTURED DATA (JSON):")
    print("-"*50)
    print(json.dumps(result['structured_response'], indent=2))
    print("="*50 + "\n")
    print("\n===============================")

### Agent to do Internal Database Retrieval
Question about game information that should be present in the local ChromaDB.
Test Query 1

In [5]:
query1 = "What platforms is 'Cyberpunk 2077' available on, and what was its original release date?"
print(f"User: {query1}")
result1 = agent.invoke(query1)
# function below has the updated dictionary parsing logic to handle both Pydantic models and standard dictionaries
print_agent_response(result1)

User: What platforms is 'Cyberpunk 2077' available on, and what was its original release date?
2026-03-30 16:54:04,069 - INFO - 
--- Processing Query: 'What platforms is 'Cyberpunk 2077' available on, and what was its original release date?' (Session: default_session) ---
2026-03-30 16:54:04,081 - INFO - Executing Node: retrieve_node
2026-03-30 16:54:04,082 - INFO - Retrieving games for query: 'What platforms is 'Cyberpunk 2077' available on, and what was its original release date?'
2026-03-30 16:54:06,280 - INFO - HTTP Request: POST https://openai.vocareum.com/v1/embeddings "HTTP/1.1 200 OK"
2026-03-30 16:54:06,424 - INFO - Executing Node: evaluate_node
2026-03-30 16:54:06,425 - INFO - Evaluating retrieval sufficiency...
2026-03-30 16:54:09,854 - INFO - HTTP Request: POST https://openai.vocareum.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-30 16:54:09,867 - INFO - Evaluation result: Sufficient=True, Reasoning: The retrieved context provides the original release date of 'Cyberpunk

### Web Search Fallback (Latest News)
This asks a question about the latest news, which should trigger the evaluator to determine the internal database lacks sufficient context, forcing a web search.
This is Query 2

In [ ]:
query2 = "What is the latest news regarding the release date of the game Grand Theft Auto 6?"
print(f"User: {query2}")
result2 = agent.invoke(query2)
print("Assess Response from agent:")
print_agent_response(result2)

User: What is the latest news regarding the release date of the game Grand Theft Auto 6?
2026-03-28 21:26:09,651 - INFO - 
--- Processing Query: 'What is the latest news regarding the release date of the game Grand Theft Auto 6?' ---
2026-03-28 21:26:09,654 - INFO - Executing Node: retrieve_node
2026-03-28 21:26:09,655 - INFO - Retrieving games for query: 'What is the latest news regarding the release date of the game Grand Theft Auto 6?'
2026-03-28 21:26:10,894 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-03-28 21:26:10,901 - INFO - Executing Node: evaluate_node
2026-03-28 21:26:10,903 - INFO - Evaluating retrieval sufficiency...
2026-03-28 21:26:13,641 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-28 21:26:13,645 - INFO - Evaluation result: Sufficient=True, Reasoning: The retrieved context provides a clear and specific release date for Grand Theft Auto VI, stating that it is scheduled to lau

### Conversational Memory
Ask a follow-up question referencing the previous query to test the agent's stateful memory (LangGraph state/history).
Test Query 3

In [6]:
query3 = "Who is the developer of that game?"
print(f"User: {query3}")
result3 = agent.invoke(query3)
print_agent_response(result3)

User: Who is the developer of that game?
2026-03-30 16:54:48,670 - INFO - 
--- Processing Query: 'Who is the developer of that game?' (Session: default_session) ---
2026-03-30 16:54:48,671 - WARNING - Deserializing unregistered type src.agent.Fact from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('src.agent', 'Fact')]
2026-03-30 16:54:48,672 - WARNING - Deserializing unregistered type src.agent.Fact from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('src.agent', 'Fact')]
2026-03-30 16:54:48,675 - INFO - Executing Node: retrieve_node
2026-03-30 16:54:49,895 - INFO - HTTP Request: POST https://openai.vocareum.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-30 16:54:49,897 - INFO - Rewritten question for retrieval: 'What is the developer of 'Cyberpunk 2077'?'
2026-03-30 16:54:49,898 - INFO - Retrieving games for query: 'What is the developer of 'Cyberpunk 2077'?'
2026-03-30 16:54:5

### Agents memory